# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt
import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [10]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-4o-mini'
openai = OpenAI()

API key looks good so far


In [63]:
admin_key = os.getenv("OPENAI_ADMIN_KEY")
def check_costs(admin_key, start_time, limit):
    cost_response = requests.get("https://api.openai.com/v1/organization/costs", 
                                 headers={"Authorization": f"Bearer {admin_key}", "Content-Type": "application/json"},
                                 params={"start_time": start_time, "limit":limit})
    return [cost["amount"] for cost in cost_response.json()["data"][0]["results"]]

In [68]:
check_costs(admin_key, int(time.time()), 1)

[{'value': 0.00146955, 'currency': 'usd'}]

In [69]:
ollama_client = OpenAI(
    base_url = 'http://localhost:11434/v1',
    api_key='ollama', # required, but unused
)
ollama_model = "gemma3n:latest"

In [22]:
# A class to represent a Webpage

# Some websites need you to use proper headers when fetching them:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:
    """
    A utility class to represent a Website that we have scraped, now with links
    """

    def __init__(self, url):
        self.url = url
        response = requests.get(url, headers=headers)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    def get_contents(self):
        return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"

In [24]:
mindy = Website("https://mindy-dossett.com")
mindy.links

['/',
 '/',
 '/about/',
 'mailto:minzhou16@gmail.com?subject=Hello%20%F0%9F%91%8B',
 '/about/',
 '/2021/04/18/collaborative-filtering-in-Java/',
 '/2021/04/18/collaborative-filtering-in-Java/',
 '/2021/04/18/collaborative-filtering-in-Java/',
 '/2021/01/24/intro-to-NN/',
 '/2021/01/24/intro-to-NN/',
 '/2021/01/24/intro-to-NN/',
 '/2020/12/31/Deploy-MDL-Lambda/',
 '/2020/12/31/Deploy-MDL-Lambda/',
 '/2020/12/31/Deploy-MDL-Lambda/',
 '/2020/12/06/Deploy-TF-MDL-EC2/',
 '/2020/12/06/Deploy-TF-MDL-EC2/',
 '/2020/12/06/Deploy-TF-MDL-EC2/',
 '/2020/11/17/imbalanced-sample/',
 '/2020/11/17/imbalanced-sample/',
 '/2020/11/17/imbalanced-sample/',
 '/2020/10/26/Car-Price-App/',
 '/2020/10/26/Car-Price-App/',
 '/2020/10/26/Car-Price-App/',
 '/2020/10/21/nlp-stopwords/',
 '/2020/10/21/nlp-stopwords/',
 '/2020/10/21/nlp-stopwords/',
 '/2020/10/19/transfer-learning/',
 '/2020/10/19/transfer-learning/',
 '/2020/10/19/transfer-learning/',
 'mailto:minzhou16@gmail.com',
 'https://www.linkedin.com/in/min

## First step: Have GPT-4o-mini figure out which links are relevant

### Use a call to gpt-4o-mini to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [25]:
link_system_prompt = "You are provided with a list of links found on a webpage. \
You are able to decide which of the links would be most relevant to include in a brochure about the company, \
such as links to an About page, or a Company page, or Careers/Jobs pages.\n"
link_system_prompt += "You should respond in JSON as in this example:"
link_system_prompt += """
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [26]:
print(link_system_prompt)

You are provided with a list of links found on a webpage. You are able to decide which of the links would be most relevant to include in a brochure about the company, such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}



In [27]:
def get_links_user_prompt(website):
    user_prompt = f"Here is the list of links on the website of {website.url} - "
    user_prompt += "please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. \
Do not include Terms of Service, Privacy, email links.\n"
    user_prompt += "Links (some might be relative links):\n"
    user_prompt += "\n".join(website.links)
    return user_prompt

In [28]:
print(get_links_user_prompt(mindy))

Here is the list of links on the website of https://mindy-dossett.com - please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. Do not include Terms of Service, Privacy, email links.
Links (some might be relative links):
/
/
/about/
mailto:minzhou16@gmail.com?subject=Hello%20%F0%9F%91%8B
/about/
/2021/04/18/collaborative-filtering-in-Java/
/2021/04/18/collaborative-filtering-in-Java/
/2021/04/18/collaborative-filtering-in-Java/
/2021/01/24/intro-to-NN/
/2021/01/24/intro-to-NN/
/2021/01/24/intro-to-NN/
/2020/12/31/Deploy-MDL-Lambda/
/2020/12/31/Deploy-MDL-Lambda/
/2020/12/31/Deploy-MDL-Lambda/
/2020/12/06/Deploy-TF-MDL-EC2/
/2020/12/06/Deploy-TF-MDL-EC2/
/2020/12/06/Deploy-TF-MDL-EC2/
/2020/11/17/imbalanced-sample/
/2020/11/17/imbalanced-sample/
/2020/11/17/imbalanced-sample/
/2020/10/26/Car-Price-App/
/2020/10/26/Car-Price-App/
/2020/10/26/Car-Price-App/
/2020/10/21/nlp-stopwords/
/2020/10/21/nlp-stopwords/
/

In [73]:
def get_links(url):
    website = Website(url)
    response = ollama_client.chat.completions.create(
        model=ollama_model,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(website)}
      ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    return json.loads(result)

In [71]:
# Anthropic has made their site harder to scrape, so I'm using HuggingFace..

huggingface = Website("https://huggingface.co")
huggingface.links

['/',
 '/models',
 '/datasets',
 '/spaces',
 '/docs',
 '/enterprise',
 '/pricing',
 '/login',
 '/join',
 'inference/get-started',
 '/spaces',
 '/models',
 '/zai-org/GLM-4.5',
 '/tencent/HunyuanWorld-1',
 '/Qwen/Qwen3-30B-A3B-Instruct-2507',
 '/black-forest-labs/FLUX.1-Krea-dev',
 '/Qwen/Qwen3-Coder-30B-A3B-Instruct',
 '/models',
 '/spaces/enzostvs/deepsite',
 '/spaces/Qwen/Qwen3-Coder-WebDev',
 '/spaces/zumjoy/Multi-Style_Video-to-Anime_Generator',
 '/spaces/smola/higgs_audio_v2',
 '/spaces/Wan-AI/Wan-2.2-5B',
 '/spaces',
 '/datasets/Kratos-AI/KAI_handwriting-ocr',
 '/datasets/Kratos-AI/airline-customersupport-englishaudio',
 '/datasets/Kratos-AI/KAI_speech-recognition-data',
 '/datasets/Kratos-AI/medical-prescription-english-audio',
 '/datasets/Kratos-AI/KAI_car-images',
 '/datasets',
 '/join',
 '/pricing#endpoints',
 '/pricing#spaces',
 '/pricing',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/allenai',
 '/facebook'

In [74]:
get_links("https://huggingface.co")

{'links': [{'type': 'about page', 'url': 'https://hugginface.co'},
  {'type': 'models page', 'url': 'https://hugginface.co/models'},
  {'type': 'datasets page', 'url': 'https://hugginface.co/datasets'},
  {'type': 'spaces page', 'url': 'https://huggingface.co/spaces'},
  {'type': 'docs page', 'url': 'https://huggingface.co/docs'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://hugginface.co/pricing'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/hugginface/'},
  {'type': 'changelog', 'url': 'https://huggingface.co/changelog'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'discord', 'url': 'https://join.discord.gg/hugginface'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT4-o

In [84]:
def get_all_details(url):
    result = "Landing page:\n"
    result += Website(url).get_contents()
    links = get_links(url)
    print("Found links:", links)
    for link in links["links"]:
        result += f"\n\n{link['type']}\n"
        try:
            result += Website(link["url"]).get_contents()
        except:
            print(link["url"])
    return result

In [85]:
print(get_all_details("https://huggingface.co"))

Found links: {'links': [{'type': 'about page', 'url': 'https://hugginface.co/'}, {'type': 'models page', 'url': 'https://hugginface.co/models'}, {'type': 'datasets page', 'url': 'https://hugginface.co/datasets'}, {'type': 'spaces page', 'url': 'https://huggingface.co/spaces'}, {'type': 'docs page', 'url': 'https://huggingface.co/docs'}, {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'}, {'type': 'careers page', 'url': 'https://apply.workable.com/hugginface/'}, {'type': 'blog', 'url': 'https://huggingface.co/blog'}, {'type': 'changelog', 'url': 'https://huggingface.co/changelog'}, {'type': 'brand', 'url': 'https://huggingface.co/brand'}, {'type': 'learn', 'url': 'https://huggingface.co/learn'}, {'type': 'discuss forum', 'url': 'https://discuss.huggingface.co/'}, {'type': 'github', 'url': 'https://github.com/huggingface'}, {'type': 'twitter', 'url': 'https://twitter.com/huggingface'}, {'type': 'linke

In [79]:
# system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
# and creates a short brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
# Include details of company culture, customers and careers/jobs if you have the information."

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
and creates a short humorous, entertaining, jokey brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
Include details of company culture, customers and careers/jobs if you have the information."


In [80]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"You are looking at a company called: {company_name}\n"
    user_prompt += f"Here are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\n"
    user_prompt += get_all_details(url)
    user_prompt = user_prompt[:5000] # Truncate if more than 5,000 characters
    return user_prompt

In [86]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'about page', 'url': 'https://hugginface.co/'}, {'type': 'careers page', 'url': 'https://apply.workable.com/hugginface/'}, {'type': 'enterprise', 'url': 'https://enterprise.hugginface.co/'}, {'type': 'pricing', 'url': 'https://pricing.hugginface.co/'}, {'type': 'docs', 'url': 'https://docs.hugginface.co/'}, {'type': 'blog', 'url': 'https://blog.hugginface.co/'}, {'type': 'brand', 'url': 'https://www.hugginface.co/brand/'}, {'type': 'join', 'url': 'https://join.huggingface.co/'}, {'type': 'status', 'url': 'https://status.hugginface.co/'}, {'type': 'github', 'url': 'https://github.com/huggingface'}, {'type': 'twitter', 'url': 'https://twitter.com/huggingface'}, {'type': 'linkedin', 'url': 'https://www.linkedin.com/company/huggingface/'}]}
https://join.huggingface.co/


'You are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\nLanding page:\nWebpage Title:\nHugging Face – The AI community building the future.\nWebpage Contents:\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nNEW\nGet started with Inference in seconds 🚀\nReachy Mini: The Open Robot for AI Builders\nWelcome Cohere on the Hub 🔥\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 1M+ models\nTrending on\nthis week\nModels\nzai-org/GLM-4.5\nUpdated\n5 days ago\n•\n8.63k\n•\n925\ntencent/HunyuanWorld-1\nUpdated\n3 days ago\n•\n9.63k\n•\n504\nQwen/Qwen3-30B-A3B-Instruct-2507\nUpdated\n3 days ago\n•\n34.9k\n•\n353\nblack-forest-labs/FLUX.1-Krea-dev\nUpdated\n2 days ago\n•\n22.4k\n•\n330\nQwen/Q

In [87]:
def create_brochure(company_name, url):
    response = ollama_client.chat.completions.create(
        model=ollama_model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [88]:
create_brochure("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'about page', 'url': 'https://hugginface.co/'}, {'type': 'models page', 'url': 'https://hugginface.co/models'}, {'type': 'datasets page', 'url': 'https://hugginface.co/datasets'}, {'type': 'spaces page', 'url': 'https://huggingface.co/spaces'}, {'type': 'docs page', 'url': 'https://huggingface.co/docs'}, {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'}, {'type': 'careers page', 'url': 'https://apply.workable.com/hugginface/'}, {'type': 'blog page', 'url': 'https://huggingface.co/blog'}, {'type': 'changelog page', 'url': 'https://huggingface.co/changelog'}, {'type': 'brand page', 'url': 'https://huggingface.co/brand'}, {'type': 'learn page', 'url': 'https://huggingface.co/learn'}, {'type': 'discuss page', 'url': 'https://discuss.hugginface.co/'}, {'type': 'github page', 'url': 'https://github.com/huggingface'}, {'type': 'twitter page', 'url': 'https://twitter.com/hu

## Hugging Face: Your AI Wingman! 🚀

**(Image: A friendly robot waving, wearing a tiny Hugging Face logo.)**

**Tired of AI being a lonely pursuit?  We get it!**  At Hugging Face, we're building the future of AI *together*.  Think of us as the biggest, friendliest AI community – a place where brilliant minds collaborate on everything from groundbreaking models to awesome apps.

**What We Do (In Plain English):**

*   **Models Galore:**  We host over **1 MILLION** pre-trained AI models!  Whether you need to generate text, images, or even code, we've got something for you.  (Seriously, a lot.)
*   **Datasets for Days:**  Need data to train your AI?  We've got **250,000+ datasets** ready to go.  From handwriting recognition to audio transcriptions, we've got you covered.
*   **Spaces - AI Apps at Your Fingertips:**  Want to *see* AI in action?  Spaces let you build and share AI applications in minutes!  From generating anime videos to creating code from descriptions, the possibilities are endless.  (Think of it as an AI app store!)
*   **Community Power:**  Join a vibrant community of AI enthusiasts, researchers, and developers.  Share your work, get feedback, and learn from the best.

**Why Choose Hugging Face?**

*   **Open Source & Collaborative:** We're all about open source.  Our tools and libraries are freely available, empowering everyone to build amazing things.
*   **Easy to Use:**  We make AI accessible.  Our user-friendly tools and libraries mean you can start building AI applications even if you're not a seasoned expert.
*   **Enterprise Ready:**  Need enterprise-grade security, support, and dedicated resources?  We've got you covered.  We help organizations like Meta, Amazon, Google, and Microsoft accelerate their AI initiatives.
*   **We're Building the Foundation:**  We're not just about today's AI; we're building the tools that will power the future.  Our libraries (Transformers, Diffusers, etc.) are used by researchers and developers worldwide.

**Career Opportunities:**

**(Image: A diverse group of people collaborating around a whiteboard covered in AI diagrams.)**

Want to join the fun?  We're always looking for talented people to help us build the future of AI!  Check out our open positions on the [Jobs page](link to jobs page).  We offer a collaborative environment, competitive benefits, and the chance to work on cutting-edge technology.

**Ready to dive in?**

[Sign Up for Free!](link to signup page)

**(Small print at the bottom):**  Hugging Face.  Making AI accessible to all.  (And having a little fun while we're at it!)

---

**Note:** I've added placeholder links where appropriate.  You'll need to replace those with the actual URLs from the Hugging Face website.  I've also added some visual elements (images) to make the brochure more appealing.  The tone is lighthearted and aims to be engaging for a broad audience.  I've also included the information about the company's culture, customers and careers/jobs as requested.





## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [89]:
def stream_brochure(company_name, url):
    stream = ollama_client.chat.completions.create(
        model=ollama_model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

In [90]:
stream_brochure("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'about page', 'url': 'https://hugginface.co/'}, {'type': 'models page', 'url': 'https://huggingface.co/models'}, {'type': 'datasets page', 'url': 'https://huggingface.co/datasets'}, {'type': 'spaces page', 'url': 'https://huggingface.co/spaces'}, {'type': 'docs page', 'url': 'https://huggingface.co/docs'}, {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'}, {'type': 'careers page', 'url': 'https://apply.workable.com/hugginface/'}, {'type': 'blog', 'url': 'https://huggingface.co/blog'}, {'type': 'changelog', 'url': 'https://huggingface.co/changelog'}, {'type': 'linkedin page', 'url': 'https://www.linkedin.com/company/huggingface/'}, {'type': 'discord', 'url': 'https://join.discord.gg/huggingface'}]}
https://join.discord.gg/huggingface


## Hugging Face: We're Making AI Accessible (and Kinda Fun!)

**(Image: A friendly cartoon robot giving a thumbs up)**

**Tired of AI that's complicated?**  So are we! At Hugging Face, we're building the future of AI, one community collaboration at a time. Think of us as the open-source clubhouse for everyone who's excited about machine learning.

**What do we do?**

*   **Models Galore:**  We host over **1 million** pre-trained AI models – from text generators to image creators.  Need a model to write poetry?  We've got it.  Want to turn your cat into a Renaissance painting?  Yep, we've got that too!
*   **Datasets for Days:**  Need data to train your own AI?  We have **250,000+ datasets** ready to go.  From handwriting recognition to audio transcription, we've got the raw materials.
*   **Spaces – Your AI Playground:**  Easily create and share AI applications with our "Spaces" platform.  Build a demo, a tool, or just a cool experiment – it's all welcome!
*   **A Thriving Community:**  Connect with fellow AI enthusiasts, researchers, and developers.  Share your work, get feedback, and learn from the best.

**Who's using Hugging Face?**

Seriously, *everyone*.  We're powering AI solutions for giants like **Meta, Amazon, Google, Intel, Microsoft, Grammarly, and even Netflix!**  (They're using us to make their content recommendations even better).  And we're working with 50,000+ other organizations.

**Why join the Hugging Face crew?**

*   **Innovation:** Be at the forefront of AI development.
*   **Impact:**  Help shape the future of technology.
*   **Community:**  Work with passionate and talented people.
*   **Growth:**  We're growing fast, and we're always looking for new team members!

**We're hiring!** Check out our open positions [here](https://huggingface.co/jobs).  We're looking for engineers, researchers, marketers, and more!

**(Image:  A collage of diverse people collaborating around a computer screen)**

**Want to learn more?**

*   **Explore our models:** [https://huggingface.co/models](https://huggingface.co/models)
*   **Dive into our datasets:** [https://huggingface.co/datasets](https://huggingface.co/datasets)
*   **Build something awesome in Spaces:** [https://huggingface.co/spaces](https://huggingface.co/spaces)
*   **Join the conversation:** [https://huggingface.co/community](https://huggingface.co/community)

**Hugging Face:  AI for all.  Let's build the future together!**

**(Small Print):**  We offer enterprise solutions for teams needing advanced security and support.  Pricing starts at $0.60/hour for GPU compute.  [https://huggingface.co/pricing](https://huggingface.co/pricing)





In [91]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'about page', 'url': 'https://hugginface.co/'}, {'type': 'pricing page', 'url': 'https://pricing.huggingface.co/'}, {'type': 'careers page', 'url': 'https://apply.workable.com/hugginface/'}, {'type': 'enterprise page', 'url': 'https://enterprise.huggingface.co/'}, {'type': 'docs page', 'url': 'https://docs.huggingface.co/'}, {'type': 'blog page', 'url': 'https://blog.huggingface.co/'}, {'type': 'community/discuss page', 'url': 'https://discuss.huggingface.co/'}, {'type': 'community/github page', 'url': 'https://github.com/huggingface'}, {'type': 'community/twitter page', 'url': 'https://twitter.com/huggingface'}, {'type': 'community/linkedin page', 'url': 'https://www.linkedin.com/company/huggingface/'}]}
https://pricing.huggingface.co/
https://enterprise.huggingface.co/
https://docs.huggingface.co/
https://blog.huggingface.co/


## Hugging Face: Your AI Adventure Starts Here! 🚀

**(Image: A friendly robot hugging a globe, overlaid with the Hugging Face logo)**

**Tired of AI that's a black box?  Welcome to Hugging Face – the AI community building the future, *together*!**

We're not just a company; we're a vibrant hub where machine learning enthusiasts, researchers, and developers collaborate to create, share, and deploy the most amazing AI models, datasets, and applications. Think of us as the GitHub for AI – but way cooler. 😎

### What We Do (And Why You Should Care):

*   **Models Galore:**  Explore a staggering **1 MILLION+** pre-trained models! From text generation to image creation, we've got something for everyone.  We're talking cutting-edge tech, constantly updated by a passionate community.  (Check out the trending models – you might find your next big thing!)
*   **Datasets for Days:**  Need data? We've got you covered with **250,000+** datasets, ready to fuel your AI projects.  Whether it's handwriting recognition, audio transcriptions, or car images, we've got the raw material.
*   **Spaces – Your AI Playground:**  Easily build and deploy interactive AI demos with our Spaces platform.  Share your creations with the world and get instant feedback!  It's like having your own personal AI showcase.
*   **Community Power:**  Join a supportive and collaborative community of over **50,000 organizations**!  Ask questions, share your work, and learn from the best in the field.  (Check out our forums – we're always happy to help!)
*   **Open Source at Heart:** We are committed to open source and building the foundation of ML tooling with the community.  Our libraries like Transformers, Diffusers, and more are used by developers worldwide.

### Who We Serve:

We're powering innovation across the board!  From **AI giants like Google, Meta, and Amazon** to **innovative startups like Grammarly and Writer**, organizations of all sizes are leveraging Hugging Face to accelerate their AI initiatives.  We're even helping non-profits make a difference!

###  Thinking of a Career with Us? 🧑‍💻

We're growing fast and looking for talented individuals to join our team!  We offer a collaborative, supportive, and fun work environment.  Plus, you'll be working on projects that are shaping the future of AI.  

**(Image:  A diverse group of people happily working together in a modern office)**

**Current openings include:** (Check out our careers page for the latest listings!)

*   Software Engineers
*   Research Scientists
*   Community Managers
*   Product Managers

###  Need AI Power?  We've Got You Covered! 💰

*   **Compute:**  Deploy your models with ease using our optimized inference endpoints.  We offer flexible pricing starting at just $0.60/hour!
*   **Enterprise Solutions:**  Get enterprise-grade security, access controls, and dedicated support for your AI projects.  Starting at $20/user/month.

**Ready to dive in?**

[Sign Up for Free!](Link to Sign Up Page)

[Explore Models](Link to Models Page)

[Browse Datasets](Link to Datasets Page)

[Check out Spaces](Link to Spaces Page)

**Follow us on:** [GitHub](Link to GitHub), [Twitter](Link to Twitter), [LinkedIn](Link to LinkedIn), [Discord](Link to Discord)

**(Small print at the bottom):**  Hugging Face –  Making AI accessible to everyone.





<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>